# Updated preprocessing


Load Doppler and Label.

Reorder Doppler to frames-first shape (N, 1, H, W) 0
optionally apply log10
Align image/label length with hf.mismatch(...) (trim to shortest)
Pad/crop frames to square target size (TARGET_SIZE=112).
optionally Load or create an ROI mask (roi_<file_idx>.npy). (similar to how its implemented in code\1_preprocessing\1_processing.ipynb, new boolean roi_on (true = create/load roi mask))

Run PCA denoising (hf.pca_denoise).

Zero values outside ROI. (if roi_on is true)

Optional temporal high-pass filter (HIGH_PASS == 'hp'; currently set to nohp, so off).
Percentile clip (BOTTOM=1, TOP=99).
ROI z-score normalization (hf.normalize_cbv_in_roi(..., 'zscore')). or just zscore and mean_divide if roi_on = false ( normalize as per code\1_preprocessing\1_processing.ipynb)
save nrmalized and preprocessed frames:
- files per normalization mode
- if highpass filter applied, labeled with _hp at the end


In [ ]:
import sys
from pathlib import Path

import yaml

# Locate repo root (same style as EDA notebooks)
repo_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "config" / "config.yml").exists()
)

# Add code directory to import path
code_dir = repo_root / "code"
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

# Load config
config_path = repo_root / "config" / "config.yml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

# Core paths + subjects
deriv_root = repo_root / config["paths"]["preprocessing"]
source_root = repo_root / config["paths"]["sourcedata"]
subjects = config["subjects"]["all"]
sample_subject = config["subjects"]["default"]

In [ ]:
import os
import importlib

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from utils import helper_functions as hf
from utils.preprocessing import standardization as stdz
from utils.preprocessing.io import (
    load_all_baseline,
    process_all_baseline_files,
    summarize_npz_collection,
)
from utils.helper_functions import get_or_create_roi_mask

---
### load subjects and plot raw fUS activity of their first session over time with label shading (blue = baseline)

In [ ]:
hf = importlib.reload(hf)

for subject in subjects:
    data_directory = source_root / subject
    data_output_dir = deriv_root / subject
    data_output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Subject: {subject}")
    print(f"  data_directory: {data_directory}")
    print(f"  data_output_dir: {data_output_dir}")

    hf.plot_fus_timecourse_with_labels(str(data_directory), sessions="first")

---
### use preprocessing/io.py baseline extraction + loading ---

In [ ]:
APPLY_LOG10 = True
LOG10_EPS = 1e-6
OVERWRITE_BASELINE = False

for subject in subjects:
    data_directory = source_root / subject
    baseline_output_dir = deriv_root / subject / "baseline_only"
    baseline_output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[{subject}] baseline extraction -> {baseline_output_dir}")

    baseline_npz_paths = process_all_baseline_files(
        data_directory=str(data_directory),
        output_dir=str(baseline_output_dir),
        overwrite=OVERWRITE_BASELINE,
        apply_log10=APPLY_LOG10,
        log10_eps=LOG10_EPS,
    )

    print(f"Saved/registered {len(baseline_npz_paths)} baseline files")

    sessions = load_all_baseline(str(baseline_output_dir))
    print(f"Loaded {len(sessions)} sessions")

    summary_df = summarize_npz_collection(baseline_npz_paths)
    display(summary_df)

    if sessions:
        s0 = sessions[0]
        frames0 = s0["frames"]
        print(f"First session_id: {s0['session_id']}")
        print(f"Frames shape: {frames0.shape}, dtype: {frames0.dtype}")
        print(f"Stage: {s0['metadata'].get('stage')}")
        print(f"did_log10: {s0['metadata'].get('did_log10')} | log10_eps: {s0['metadata'].get('log10_eps')}")

---
### Load Baseline Data

Load baseline sessions for model training.

In [ ]:
for subject in subjects:
    baseline_output_dir = deriv_root / subject / "baseline_only"
    baseline_sessions = load_all_baseline(str(baseline_output_dir))

    if len(baseline_sessions) == 0:
        print(f"\n[{subject}] No baseline sessions found")
        continue

    all_npz_paths = sorted(str(p) for p in baseline_output_dir.glob("baseline_*.npz"))
    summary_df = summarize_npz_collection(all_npz_paths)

    print(f"\nBaseline Data Summary: {subject}")
    print(f"  Total sessions: {len(baseline_sessions)}")
    total_frames = sum(s["frames"].shape[0] for s in baseline_sessions)
    print(f"  Total baseline frames: {total_frames:,}")

    spatial_shapes = [s["frames"].shape[1:] for s in baseline_sessions]
    print(f"  Spatial dimensions: {set(spatial_shapes)}")

    frame_counts = [s["frames"].shape[0] for s in baseline_sessions]
    print(
        f"  Frames per session: min={min(frame_counts)}, max={max(frame_counts)}, "
        f"mean={np.mean(frame_counts):.0f}, std={np.std(frame_counts):.0f}"
    )

    first_session = baseline_sessions[0]
    print(f"\n  First session ({first_session['session_id']}):")
    print(f"    Frames: {first_session['frames'].shape[0]}")
    print(f"    Shape: {first_session['frames'].shape}")
    print(f"    Dtype: {first_session['frames'].dtype}")
    print(
        f"    Value range: [{first_session['frames'].min():.2f}, {first_session['frames'].max():.2f}]"
    )

    print("\n  Stage summary:")
    display(summary_df[["filename", "session_id", "stage", "schema_version", "did_log10", "log10_eps", "error"]])